## Sanity check: `fct_monthly_schedule_rt_route_direction_summary`

Check quartet, now that left join adds in prediction accuracy metrics.

Prediction accuracy is calculated for the month - day_type, then merged in with a left join.
Other tu metrics (tu completeness, etc) is available for all dates, but prediction accuracy only available for a week.

* If only selecting schedule + trip updates, is it ok to have duplicate rows for vp?
   * vp can show up with trips, or vp doesn't have trips
   * does this make sense for a route-direction? Yes, some trips along that route can have vp info, some trips won't.
* Left join in the dbt model makes sense, the trip update info, if it's available for that route, will get attached. Even if there are 2 records, the duplicate is coming from vp, not from tu.

In [1]:
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import report_utils
from gtfs_curator_utils import utils
from shared_vars import PREDICTIONS_GCS, RT_MSA_DICT

credentials, _ = google.auth.default()

/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-pack

In [2]:
# play with dropna vp_name vs without
# are these cases of left join ok in dbt model?

In [3]:
DOWNLOADED_DICT = RT_MSA_DICT.dbt_model_downloads
PROCESSED_DICT = RT_MSA_DICT.rt_schedule_models
filename = DOWNLOADED_DICT.weekday_route_direction_grain

In [4]:
df1 = pd.read_parquet(
    f"{PREDICTIONS_GCS}{filename}.parquet",
    filesystem=gcsfs.GCSFileSystem(),
).dropna(subset="n_tu_trips_prediction_accuracy")

In [5]:
df2 = df1.dropna(subset="vp_name")

In [6]:
group_cols = [
    "month_first_day",
    "day_type",
    "tu_base64_url",
    "tu_name",
    "schedule_base64_url",
    "schedule_name",
    "route_name",
    "direction_id", 
]

In [7]:
df1[group_cols].drop_duplicates().shape

(30566, 8)

In [8]:
df2[group_cols].shape

(30410, 8)

In [9]:
df_dupe = (
    df1.groupby(group_cols)
    .agg({"prediction_error_sec_array": "count"})
    .reset_index()
    .query('prediction_error_sec_array > 1')
)[group_cols]

In [10]:
# Different number of trips can show up with vp, 
# but tu prediction accuracy is present for the month, because of how aggregation is calculated
pd.merge(
    df1,
    df_dupe,
    on = group_cols,
    how = "inner"
)[group_cols + ["vp_name", "appeared_in_vp", "n_vp_trips", "prediction_error_sec_array"]].sort_values(group_cols)


,month_first_day,day_type,tu_base64_url,tu_name,schedule_base64_url,schedule_name,route_name,direction_id,vp_name,appeared_in_vp,n_vp_trips,prediction_error_sec_array
3299,2026-09-01,Saturday,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,Bay Area 511 Regional TripUpdates,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,Bay Area 511 Regional Schedule,MA:219__219 Tiburon - Strawberry,0,Bay Area 511 Regional VehiclePositions,True,49.0,"[-11.3, 0.0, 14.0, 32.3, 45.0, 60.0, 78.0, 98...."
3302,2026-09-01,Saturday,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,Bay Area 511 Regional TripUpdates,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,Bay Area 511 Regional Schedule,MA:219__219 Tiburon - Strawberry,0,None,False,NaN,"[-11.3, 0.0, 14.0, 32.3, 45.0, 60.0, 78.0, 98...."
3429,2026-09-01,Saturday,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,Bay Area 511 Regional TripUpdates,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,Bay Area 511 Regional Schedule,MA:219__219 Tiburon - Strawberry,1,Bay Area 511 Regional VehiclePositions,True,49.0,"[-17.7, 0.0, 17.0, 24.0, 36.0, 56.0, 81.0, 97...."
3431,2026-09-01,Saturday,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,Bay Area 511 Regional TripUpdates,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,Bay Area 511 Regional Schedule,MA:219__219 Tiburon - Strawberry,1,None,False,NaN,"[-17.7, 0.0, 17.0, 24.0, 36.0, 56.0, 81.0, 97...."
220,2026-09-01,Saturday,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,Bay Area 511 Regional TripUpdates,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,Bay Area 511 Regional Schedule,SC:202__202 2nd & Santa Clara - Winchester,0,None,False,NaN,"[-28.3, -14.3, 11.7, 34.0, 42.0, 79.0, 161.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
2998,2026-11-01,Weekday,aHR0cHM6Ly9zdW5saW5lYnVzLmNvbS9ndGZzLXJ0L3RyaX...,SunLine GMV Trip Updates,aHR0cHM6Ly9zdW5saW5lYnVzLmNvbS9ndGZz,SunLine GMV Schedule,6826__4 Route 4,1,None,False,NaN,"[-370.0, -254.0, -135.7, -98.0, -67.3, -23.0, ..."
2606,2026-11-01,Weekday,aHR0cHM6Ly9zdW5saW5lYnVzLmNvbS9ndGZzLXJ0L3RyaX...,SunLine GMV Trip Updates,aHR0cHM6Ly9zdW5saW5lYnVzLmNvbS9ndGZz,SunLine GMV Schedule,6827__5 Route 5,0,SunLine GMV Vehicle Positions,True,88.0,"[-323.0, -217.7, -110.7, -75.0, -45.7, -3.0, 1..."
2607,2026-11-01,Weekday,aHR0cHM6Ly9zdW5saW5lYnVzLmNvbS9ndGZzLXJ0L3RyaX...,SunLine GMV Trip Updates,aHR0cHM6Ly9zdW5saW5lYnVzLmNvbS9ndGZz,SunLine GMV Schedule,6827__5 Route 5,0,None,False,NaN,"[-323.0, -217.7, -110.7, -75.0, -45.7, -3.0, 1..."
2999,2026-11-01,Weekday,aHR0cHM6Ly9zdW5saW5lYnVzLmNvbS9ndGZzLXJ0L3RyaX...,SunLine GMV Trip Updates,aHR0cHM6Ly9zdW5saW5lYnVzLmNvbS9ndGZz,SunLine GMV Schedule,6829__6 Route 6,0,SunLine GMV Vehicle Positions,True,135.0,"[-290.7, -191.0, -97.7, -68.0, -42.7, -6.0, 14..."
